1. 加载数据集。这个数据集与常规不同，是一个python的二进制文件

In [ ]:
import pickle
import numpy as np
TRAIN_PATH = "datasets/train"
with open(TRAIN_PATH, "rb") as f:
    dictionary = pickle.load(f, encoding="bytes")
print(dictionary.keys())
raw_data = dictionary[b"data"]
images = raw_data.reshape(-1, 3, 32, 32)
# label值
labels = np.array(dictionary[b"fine_labels"])

print(images.shape) # (50000, 3, 32, 32)


打印其中一个图片看看效果

In [ ]:
# 打印第一个图片看看图片和label
import matplotlib.pyplot as plt
plt.imshow(images[0].transpose(1, 2, 0))
plt.axis("off")
plt.title(f"Label: {labels[0]}")
plt.show()


准备训练和验证数据集
1. 数据归一化处理
2. 分割数据集

In [ ]:
#首先看看二者大小是否一致

print("特征数据形状:", images.shape)
print("目标类 形状:", labels.shape)
images = images.astype(np.float32) / 255.0
# 切分
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)
print("训练集图片:", X_train.shape)
print("验证集图片:", X_val.shape)

print("训练集标签:", y_train.shape)
print("验证集标签:", y_val.shape)


使用pytorch

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
# device = torch.device("mps" if torch.cuda.is_available() else "cpu")
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print(device)

定义数据集合类和数据加载类

In [ ]:
class CIFARDataset(Dataset):

    def __init__(self, images, labels,transform):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.transform = transform
    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):

        image = self.images[index]
        label = self.labels[index]

        if self.transform:
            image = self.transform(image)

        return image, label

# =========================
# 数据增强
# =========================
from torchvision import transforms
#训练集转换+增强
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor()
])
#验证集只转换，不增强
val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])


train_dataset = CIFARDataset(
    X_train,
    y_train,
    transform=train_transform
)

val_dataset = CIFARDataset(
    X_val,
    y_val,
    transform=val_transform
)
image, label = train_dataset[0]

print(image.shape)
print(label)
print("训练样本:", len(train_dataset))
print("验证样本:", len(val_dataset))

定义批处理和数据加载对象

In [ ]:
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

定义CNN网络

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # block1
            nn.Conv2d(3,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),


            # block2
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2),


            # block3
            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.MaxPool2d(2)

        )


        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(256*4*4,512),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(512,100)

        )


    def forward(self,x):

        x=self.features(x)

        x=self.classifier(x)

        return x

实例化cnn

In [ ]:
model = CNN()
#设置使用cuda

model = model.to(device)
# 确保模型在 GPU
print(model)

In [ ]:
# 拿到第一批batch的图片和标签数据
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)
print(images.shape)
print(labels.shape)


In [ ]:
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)
output = model(images)

print("Input shape:", images.shape)
print("Output shape:", output.shape)

定义损失函数和优化器

In [ ]:
#使用交叉熵损失函数
criterion = nn.CrossEntropyLoss()
# 怎么根据错误修改模型参数
# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr=0.001
# )

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4
)

In [ ]:
# 查看下验证数据集
print("train labels:", torch.unique(train_dataset.labels))
print("val labels:", torch.unique(val_dataset.labels))


In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print(outputs[0])
print(torch.softmax(outputs[0], dim=0))

In [ ]:
#定义最佳准确率初始值
best_accuracy = 0.0

train_losses = []
train_accuracies = []
val_accuracies = []


# import os
# model_path = "best_model.pth"
#
# if os.path.exists(model_path):
#     model.load_state_dict(
#         torch.load(model_path, map_location=device)
#     )
#     print("加载模型完成。。。")
# else:
#     print("未发现模型文件. 使用当前模型参数.")


# 整个训练集看10遍
num_epochs = 100
for epoch in range(num_epochs):

    # 设定模型为训练模式
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0
    # print(model.conv1.weight.mean())
    for images, labels in train_loader:
        # 转换成GPU tensor
        images = images.to(device)
        labels = labels.to(device)


        # 清除先前的前向传播
        optimizer.zero_grad()

        # 前向传播
        outputs = model(images)

        # 计算损失
        loss = criterion(outputs, labels)

        # 后向传播：计算梯度
        loss.backward()

        # 更新参数
        optimizer.step()

        # 累计这一轮batch的所有损失值
        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        # 累加这个批次的数量
        total += labels.size(0)
        # 累加这个批次的计算正确的数量
        correct += (predicted == labels).sum().item()
    # print(model.conv1.weight.mean())
    # 第n遍全数据集跑一遍的训练损失
    train_loss = running_loss / len(train_loader)
    # 第n遍全数据集跑一遍的训练精确度
    train_accuracy = 100 * correct / total

     # =========================
    # 2. 验证模式
    # =========================
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            # 转换成 GPU Tensor
            images = images.to(device)
            labels = labels.to(device)

            # 前向传播
            outputs = model(images)

            # 获取预测类别
            _, predicted = torch.max(outputs, 1)

            # 累加数量
            total += labels.size(0)

            # 累加正确数量
            correct += (predicted == labels).sum().item()

    # 计算验证集准确率
    validation_accuracy = 100 * correct / total

    # =========================
    # 3. 保存最佳模型
    # =========================
    if validation_accuracy > best_accuracy:

        best_accuracy = validation_accuracy

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print(">>> 保存新的最佳模型！")


    # =========================
    # 4. 打印结果
    # =========================
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Accuracy: {train_accuracy:.2f}% "
        f"Val Accuracy: {validation_accuracy:.2f}%"
    )

    # 将每次epoch的数据加到列表中
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(validation_accuracy)
print(f"最佳验证集准确率为: {best_accuracy:.2f}%")

查看训练曲线

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_accuracies, label="训练准确率")
plt.plot(val_accuracies, label="验证准确率")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.show()